In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows의 경우
plt.rcParams['axes.unicode_minus'] = False  # 음수 기호 깨짐 방지

In [65]:
# 작물 교체
crop = "상추"
df = pd.read_csv(f"../data/{crop}_월차낼게요.csv", encoding="cp949")


In [66]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272381 entries, 0 to 272380
Data columns (total 35 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   year              272381 non-null  int64  
 1   week              272381 non-null  int64  
 2   week_start        272381 non-null  object 
 3   품종코드              272381 non-null  int64  
 4   등급코드              272381 non-null  int64  
 5   직팜산지코드            272381 non-null  int64  
 6   총거래량(kg)          272381 non-null  float64
 7   일평균기온             272381 non-null  float64
 8   최고기온              272381 non-null  float64
 9   최저기온              272381 non-null  float64
 10  평균상대습도            272381 non-null  float64
 11  강수량(mm)           272381 non-null  float64
 12  1시간최고강수량(mm)      272381 non-null  float64
 13  holiday_flag      272381 non-null  int64  
 14  holiday_score     272381 non-null  float64
 15  grow_score        272381 non-null  float64
 16  평균단가(원)           27

In [67]:
print("✅ 결측치 개수:\n", df.isna().sum())
print("✅ 전체 결측치 합:", df.isna().sum().sum())
print("✅ 중복 행 수:", df.duplicated().sum())


✅ 결측치 개수:
 year                0
week                0
week_start          0
품종코드                0
등급코드                0
직팜산지코드              0
총거래량(kg)            0
일평균기온               0
최고기온                0
최저기온                0
평균상대습도              0
강수량(mm)             0
1시간최고강수량(mm)        0
holiday_flag        0
holiday_score       0
grow_score          0
평균단가(원)             0
일평균기온_t-1           0
최고기온_t-1            0
최저기온_t-1            0
평균상대습도_t-1          0
강수량(mm)_t-1         0
1시간최고강수량(mm)_t-1    0
일평균기온_t-2           0
최고기온_t-2            0
최저기온_t-2            0
평균상대습도_t-2          0
강수량(mm)_t-2         0
1시간최고강수량(mm)_t-2    0
일평균기온_t-3           0
최고기온_t-3            0
최저기온_t-3            0
평균상대습도_t-3          0
강수량(mm)_t-3         0
1시간최고강수량(mm)_t-3    0
dtype: int64
✅ 전체 결측치 합: 0
✅ 중복 행 수: 5


In [68]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("🔢 수치형 컬럼:", numerical_cols)
print("🔤 범주형 컬럼:", categorical_cols)


🔢 수치형 컬럼: ['year', 'week', '품종코드', '등급코드', '직팜산지코드', '총거래량(kg)', '일평균기온', '최고기온', '최저기온', '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)', 'holiday_flag', 'holiday_score', 'grow_score', '평균단가(원)', '일평균기온_t-1', '최고기온_t-1', '최저기온_t-1', '평균상대습도_t-1', '강수량(mm)_t-1', '1시간최고강수량(mm)_t-1', '일평균기온_t-2', '최고기온_t-2', '최저기온_t-2', '평균상대습도_t-2', '강수량(mm)_t-2', '1시간최고강수량(mm)_t-2', '일평균기온_t-3', '최고기온_t-3', '최저기온_t-3', '평균상대습도_t-3', '강수량(mm)_t-3', '1시간최고강수량(mm)_t-3']
🔤 범주형 컬럼: ['week_start']


In [69]:
import numpy as np
from sklearn.preprocessing import RobustScaler

# 복사본 생성
df_scaled = df.copy()

# 🎯 로그 변환 + RobustScaler: 총거래량(kg)
df_scaled['총거래량(kg)'] = np.log1p(df_scaled['총거래량(kg)'])

# ❗제외할 컬럼
target = '평균단가(원)'
exclude_cols = ['year', 'week', '품종코드', '등급코드', '직팜산지코드', target]

# 수치형 컬럼 중 제외대상 뺀 컬럼들 추출
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [col for col in numerical_cols if col not in exclude_cols]

# RobustScaler 전체 적용
scaler = RobustScaler()
df_scaled[scale_cols] = scaler.fit_transform(df_scaled[scale_cols])


In [70]:
onehot_cols = ['품종코드', '등급코드', '직팜산지코드']
df_encoded = pd.get_dummies(df_scaled, columns=onehot_cols, dtype=int)


In [71]:
print("🎯 최종 행/열 크기:", df_encoded.shape)
df_encoded.head()


🎯 최종 행/열 크기: (272381, 202)


,year,week,week_start,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),...,직팜산지코드_1158,직팜산지코드_1159,직팜산지코드_1162,직팜산지코드_1163,직팜산지코드_1164,직팜산지코드_1165,직팜산지코드_1166,직팜산지코드_1167,직팜산지코드_1168,직팜산지코드_1169
0,2018,13,2018-03-26,0.140411,-0.123945,-0.097902,-0.154696,-0.419689,-0.009524,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2018,13,2018-03-26,0.119044,-0.047996,-0.013986,-0.254144,-0.689119,-0.009524,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2018,13,2018-03-26,-0.017080,-0.033755,-0.153846,0.066298,-0.538860,-0.009524,0.0,...,0,0,0,0,0,0,0,0,0,0
3,2018,13,2018-03-26,0.403827,-0.110970,-0.041958,-0.292818,-0.321244,-0.009524,0.0,...,0,0,0,0,0,0,0,0,0,0
4,2018,13,2018-03-26,0.775159,-0.156118,-0.097902,-0.320442,0.180484,-0.009524,0.0,...,0,0,0,0,0,0,0,0,0,0


In [72]:
df_encoded.to_csv(f"../data/{crop}_월차_학습용.csv", index=False, encoding='cp949')
print(f"✅ 저장 완료")

✅ 저장 완료
